In [43]:
import importlib
from pathlib import Path

import pandas as pd
from cuery import Field, Prompt, Response, Task, pprint
from cuery.utils import dedent
import fow

# Data

In [59]:
fp = fow.pathto("data/processed/ine_dirce/ine_dirce_aggregated_by_activity.parquet")
df = pd.read_parquet(fp)
df = df.rename(columns={"Division": "sector", "Actividad principal": "subsector"})
df

,sector,subsector,Estimated_Employees_2024,Estimated_Employees_pct,Median_YoY_Growth_pct,Growth_2020_2024_pct,Total_2024,Total_2023,Total_2022,Total_2021,...,Estrato_De 50 a 99_pct,Estrato_De 100 a 199_pct,Estrato_De 200 a 249_pct,Estrato_De 250 a 999_pct,Estrato_De 1000 a 4999_pct,Estrato_De 5000 o más asalariados_pct,Condicion_Sociedades anónimas_abs,Condicion_Sociedades de responsabilidad limitada_abs,Condicion_Sociedades anónimas_pct,Condicion_Sociedades de responsabilidad limitada_pct
0,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,216.0,0.00,-12.0,-61.1,21,23,44,52,...,4.76,0.00,0.00,0.00,0.00,0.00,7,14,33.33,66.67
1,"05 Extracción de antracita, hulla y lignito",052 Extracción de lignito,2.0,0.00,0.0,-33.3,2,3,3,3,...,0.00,0.00,0.00,0.00,0.00,0.00,2,0,100.00,0.00
2,06 Extracción de crudo de petróleo y gas natural,061 Extracción de crudo de petróleo,16.0,0.00,16.7,100.0,12,6,4,5,...,0.00,0.00,0.00,0.00,0.00,0.00,9,3,75.00,25.00
3,06 Extracción de crudo de petróleo y gas natural,062 Extracción de gas natural,8.0,0.00,-50.0,0.0,2,2,0,1,...,0.00,0.00,0.00,0.00,0.00,0.00,0,2,0.00,100.00
4,07 Extracción de minerales metálicos,071 Extracción de minerales de hierro,84.0,0.00,-17.0,-52.6,9,11,13,16,...,0.00,0.00,0.00,0.00,0.00,0.00,3,6,33.33,66.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,94 Actividades asociativas,942 Actividades sindicales,0.0,0.00,0.0,0.0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN
248,94 Actividades asociativas,949 Otras actividades asociativas,49.0,0.00,25.0,100.0,2,3,2,1,...,0.00,0.00,0.00,0.00,0.00,0.00,0,2,0.00,100.00
249,"95 Reparación de ordenadores, efectos personal...",951 Reparación de ordenadores y equipos de com...,14968.0,0.13,-2.8,-12.6,2153,2097,2324,2387,...,0.51,0.19,0.09,0.14,0.09,0.00,31,2122,1.44,98.56
250,"95 Reparación de ordenadores, efectos personal...",952 Reparación de efectos personales y artícul...,14278.0,0.12,-1.0,-9.3,3328,3336,3592,3604,...,0.21,0.06,0.03,0.03,0.03,0.00,31,3297,0.93,99.07


# Response formats

In [37]:
class JobTask(Response):
    """Individual task within an occupation."""

    task_name: str = Field(
        description="Name of the specific task or job to be done (less than 80 characters)",
        min_length=10,
        max_length=80,
    )

    task_automation_potential: int = Field(
        description="A score from 1 to 10 indicating this specific task's potential for automation",
        ge=0,
        le=10,
    )

    current_products: list[str] = Field(
        description="List of current software products/tools used to perform this task",
        min_length=0,
        max_length=8,
        default=[],
    )

    task_description: str = Field(
        description="Detailed description of what this task involves (less than 300 characters)",
        min_length=30,
        max_length=300,
    )

    task_automation_reason: str = Field(
        description="A short explanation of why this task is automatable with the given score",
        min_length=20,
        max_length=300,
    )

In [38]:
class Job(Response):
    """An occupation/job role with its associated tasks."""

    job_name: str = Field(
        description="Name of the occupation/job role (less than 60 characters)",
        min_length=5,
        max_length=60,
    )

    tasks: list[JobTask] = Field(
        description="List of specific tasks performed in this occupation",
        min_length=6,
        max_length=10,
    )


class Jobs(Response):
    """List of occupations for a sector/subsector."""

    occupations: list[Job] = Field(
        description="List of all occupations/job roles in this sector/subsector",
        min_length=7,
        max_length=20,
    )

# Prompt

In [ ]:
SYS_PROMPT = """
You're an analyst at the Spanish 'Instituto Nacional de Estadística' (INE) analyzing
data from its 'Directorio Central de Empresas' (DIRCE). Your objective is to analyze 
groups of companies, identified by a sector ('Sector') and a corresponding main activity
('Subsector') in order to identify relevant occupations and their specific automatable tasks.
Both 'Sector' and 'Subsector' are provided in Spanish and may
include numeric IDs that you can ignore if you don't understand them. Always respond in English.
Only consider tasks that are computer- or paper-based and can be automated by AI using software
(don't include tasks automatable by robots or other physical means). For each task, also identify
the current software products or tools commonly used to perform that task.
"""

USR_PROMPT = """
Please analyze the following sector:

1. First, identify 7-20 relevant occupations/job roles that would typically exist in this sector/subsector (aim for at least 10)\n
2. For EACH occupation, provide 6-10 specific automatable tasks they perform (aim for at least 7-8 tasks)

Structure your response hierarchically:

- Occupation 1: [name]\n
  - Task: [task name, description, automation score 1-10, automation reason, current tools/products used]\n
  - Task: [task name, description, automation score 1-10, automation reason, current tools/products used]\n
  - Task: [task name, description, automation score 1-10, automation reason, current tools/products used]\n
  - ... (continue for 6-10 tasks total for this occupation)\n

  - Occupation 2: [name]\n
  - Task: [task name, description, automation score 1-10, automation reason, current tools/products used]\n
  - Task: [task name, description, automation score 1-10, automation reason, current tools/products used]\n
  - ... (continue for 6-10 tasks total for this occupation)\n
- ... (continue for all 7-20 occupations)\n

IMPORTANT

1. You MUST identify between 7-20 occupations (preferably 10+)\n
2. Each occupation MUST have between 6-10 tasks (preferably 7-8)\n
3. For each task, you MUST include:\n
   - Automation potential score (1-10)\n
   - Current software products/tools used to perform the task\n
   - Explanation of why it can be automated\n
4. Think broadly about all roles in the sector - from entry-level to senior positions\n

Sector: {{sector}}
Subsector: {{subsector}}
"""

prompt = Prompt(
    messages=[
        {"role": "system", "content": dedent(SYS_PROMPT)},
        {"role": "user", "content": dedent(USR_PROMPT)},
    ],  # type: ignore
    required=["sector", "subsector"],
)

# Task

In [52]:
task = Task(prompt=prompt, response=Jobs)
pprint(task)

╭──────────────────────────────────── TASK ────────────────────────────────────╮
│                                                                              │
│ ╭───────────────────────────────── PROMPT ─────────────────────────────────╮ │
│ │                                                                          │ │
│ │  Required: ['sector', 'subsector']                                       │ │
│ │                                                                          │ │
│ │ ╭─────────────────────────────── SYSTEM ───────────────────────────────╮ │ │
│ │ │ You're an analyst at the Spanish 'Instituto Nacional de Estadística' │ │ │
│ │ │ (INE) analyzing data from its 'Directorio Central de Empresas'       │ │ │
│ │ │ (DIRCE). Your objective is to analyze  groups of companies,          │ │ │
│ │ │ identified by a sector ('Sector') and a corresponding main activity  │ │ │
│ │ │ ('Subsector') in order to identify relevant occupations and their    │ │ │
│ │ │ specific automatable t

# Run

In [ ]:
# Configure run
# model = "openai/gpt-3.5-turbo"
model = "openai/gpt-4.1-mini"
# model = "google/gemini-2.5-pro-preview-05-06"
output_fp = fp = fow.pathto("data/processed/ine_dirce/ai_tasks_enriched.parquet")
n_sectors = 1

In [ ]:
sample = df.iloc[:n_sectors]
result = await task(sample, model=model, n_concurrent=100)

In [71]:
jobs_df = result.to_pandas()
jobs_df

,sector,subsector,job_name,tasks
0,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Geologist,[task_name='Analyzing geological data' task_au...
1,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Mining Engineer,[task_name='Designing mining plans' task_autom...
2,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Environmental Specialist,[task_name='Conducting environmental impact as...
3,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Safety Officer,[task_name='Conducting safety audits' task_aut...
4,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Machine Operator,[task_name='Operating mining machinery control...
5,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Logistics Coordinator,[task_name='Scheduling equipment and transport...
6,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Maintenance Technician,[task_name='Diagnosing equipment faults' task_...
7,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Administrative Assistant,[task_name='Managing correspondence and emails...


In [ ]:
# Flatten the tasks manually
records = []
for job, tasks in zip(jobs_df.job_name, jobs_df.tasks):
    for task in tasks:
        record = {"job_name": job} | task.to_dict()
        records.append(record)

tasks_df = pd.DataFrame.from_records(records)
tasks_df

,job_name,task_name,task_automation_potential,current_products,task_description,task_automation_reason
0,Geologist,Analyzing geological data,7,"[ArcGIS, RockWorks]",Examine and interpret geological data to locat...,Data analysis and pattern recognition can be a...
1,Geologist,Mapping coal deposits,6,"[AutoCAD, ArcGIS]",Create geological maps showing the location an...,Mapping can be automated with GIS software and...
2,Geologist,Sampling and testing coal quality,5,"[LabVIEW, Excel]",Collect coal samples and conduct quality tests...,Sample analysis can be partially automated wit...
3,Geologist,Reporting geological findings,8,"[Microsoft Word, LaTeX]",Compile and present geological data in detaile...,Report generation can be automated using AI-dr...
4,Geologist,Monitoring environmental regulations,7,"[Compliance 360, Intelex]",Track and ensure compliance with environmental...,Automation aids in monitoring regulatory chang...
5,Geologist,Updating geological databases,8,"[SQL Server, PostgreSQL]",Maintain and update databases storing geologic...,Database management tasks can be automated usi...
6,Mining Engineer,Designing mining plans,7,"[Surpac, MineSight]",Develop detailed plans for coal extraction ope...,AI can assist in optimizing designs based on g...
7,Mining Engineer,Calculating resource reserves,8,"[Mine2-4D, Geovia Surpac]",Estimate the quantity and quality of coal rese...,Resource estimation uses complex data suited f...
8,Mining Engineer,Monitoring mine safety,6,"[SafetyCulture, SmartCap]",Ensure safety protocols and hazard identificat...,Automation supports real-time safety monitorin...
9,Mining Engineer,Preparing technical reports,7,"[Microsoft Word, Adobe Acrobat]","Document mining plans, progress, and technical...",Report writing can be assisted through documen...


In [83]:
joined = jobs_df.merge(tasks_df, how="left", on="job_name").drop(columns="tasks")
joined

,sector,subsector,job_name,task_name,task_automation_potential,current_products,task_description,task_automation_reason
0,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Geologist,Analyzing geological data,7,"[ArcGIS, RockWorks]",Examine and interpret geological data to locat...,Data analysis and pattern recognition can be a...
1,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Geologist,Mapping coal deposits,6,"[AutoCAD, ArcGIS]",Create geological maps showing the location an...,Mapping can be automated with GIS software and...
2,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Geologist,Sampling and testing coal quality,5,"[LabVIEW, Excel]",Collect coal samples and conduct quality tests...,Sample analysis can be partially automated wit...
3,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Geologist,Reporting geological findings,8,"[Microsoft Word, LaTeX]",Compile and present geological data in detaile...,Report generation can be automated using AI-dr...
4,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Geologist,Monitoring environmental regulations,7,"[Compliance 360, Intelex]",Track and ensure compliance with environmental...,Automation aids in monitoring regulatory chang...
5,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Geologist,Updating geological databases,8,"[SQL Server, PostgreSQL]",Maintain and update databases storing geologic...,Database management tasks can be automated usi...
6,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Mining Engineer,Designing mining plans,7,"[Surpac, MineSight]",Develop detailed plans for coal extraction ope...,AI can assist in optimizing designs based on g...
7,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Mining Engineer,Calculating resource reserves,8,"[Mine2-4D, Geovia Surpac]",Estimate the quantity and quality of coal rese...,Resource estimation uses complex data suited f...
8,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Mining Engineer,Monitoring mine safety,6,"[SafetyCulture, SmartCap]",Ensure safety protocols and hazard identificat...,Automation supports real-time safety monitorin...
9,"05 Extracción de antracita, hulla y lignito",051 Extracción de antracita y hulla,Mining Engineer,Preparing technical reports,7,"[Microsoft Word, Adobe Acrobat]","Document mining plans, progress, and technical...",Report writing can be assisted through documen...


In [84]:
joined.to_parquet(output_fp)